## PCM_pix — clean runner notebook

Этот ноутбук — **тонкий сценарий**:
- грузит данные из `data/`
- грузит/обучает суррогаты (legacy или new)
- оценивает качество суррогатов
- запускает оптимизацию (PSO / DE / PSO→DE)
- сохраняет результаты в `outputs/<run_name>/`

Старые большие ноутбуки остаются как архив (`3rd art_...`, `to_server_arch-...`).

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

# --- CONFIG (меняй здесь) ---
CFG = {
    "run_name": "2026_03_12",
    
    # surrogate
    "ann_data_dir": "data/ann_models", #где лежат уже тренированные модели
    "adb_data_dir": "data",
    "adb_data":["Sb2Se3 - amorphous_mesh_sbse_2025.txt", "Sb2Se3 - crystal_mesh12_sbse_2025.txt",], #данные из люма
    "wl": 1.55e-6,  # фильтруем датасет по длинам волн
   
    "train_load_mode": "load", # train/load
    "device": "cpu", #cpu/cuda
    "epochs": 10000,
    "lr": 1e-3,
    "qa_n": 5000, # количество точек для оценки суррогатов

    "preset_dir": "data/preset/",

    # fabrication settings
    "Nn": 11,
    "b_min_m": 50e-9, #fabrication limit

    #optimization

    "opt_mode": "hybrid_pso_de",      # "preset" | "pso" | "pso_until" | "de_full" | "hybrid_pso_de"
    
    
    "preset_name": "pos_server_2026_03_04_night",

    # hyperopt modes
    # - legacy ключ (общий), оставлен для совместимости
    # - новые ключи позволяют управлять PSO и DE независимо
    #"hyperopt_mode": "run",        # legacy: "auto" | "use_saved" | "run"
    "pso_hyperopt_mode": "use_saved",    # "auto" | "use_saved" | "run"
    "de_hyperopt_mode": "use_saved",     # "auto" | "use_saved" | "run"

    # PSO
    "pso_threshold": 4,
    "pso_max_restarts": 20,
    "pso_n_particles": 3500,
    "pso_iters": 350,
    "pso_c1": 1.4,
    "pso_c2": 0.65,
    "pso_w": 0.6,
    
    "hyperopt_pso_trials": 100,
    "hyperopt_pso_repeats": 2,
    "hyperopt_pso_refine_topk": 2,
    "hyperopt_pso_refine_repeats": 1,


    # DE (full compatibility)
    "de_init_mode": "init_ar",    # "init_ar" - через начальное состояние, которое генерируется из исходника с шумом 
    "de_init_spread": 0.1, # 10% от исходника
   #"de_init_mode": "x0",    # "x0" - через начальное состояние
    "de_init_N": 10000,
    "de_mutation": 0.99,
    "de_recombination": 0.1,
    "de_maxiter": 50000,
    "de_popsize": 60,
    "de_tol": 1e-8,
    "de_atol": 1e-3,
    "de_polish": True,
    "de_callback_every": 1000,
    "de_use_linear_constraint": True,


}

# optional: reproducibility seed
SEED = round(np.random.random()*100000)
np.random.seed(SEED)
print(SEED)

67266


In [2]:
from pcm_pix.run import init_notebook_run
run, logger = init_notebook_run(CFG, notebook_name="main")

2026-03-29 17:34:48,098 | INFO | run_dir=/media/slava/Data/git/PCM_pix/outputs/2026_03_12
2026-03-29 17:34:48,141 | INFO | main started


In [3]:
# --- Dataset + surrogates + Metrics ---
from pcm_pix.nn_surrogate import build_ANN
df, data_0, data_1, X_0, y_0, X_1, y_1, sur0, sur1, qa_am, qa_cr = build_ANN(CFG, run)
qa_am, qa_cr  # чтобы увидеть словари метрик в output

2026-03-29 17:35:06,986 | INFO | loading surrogates from outputs/2026_03_12/models
2026-03-29 17:35:07,081 | INFO | surrogates OK
2026-03-29 17:35:07,082 | INFO | dataset: df=98390 data_0=60360 data_1=38030
2026-03-29 17:35:07,082 | INFO | X_0=(60360, 3) y_0=(60360, 4) | X_1=(38030, 3) y_1=(38030, 4)
2026-03-29 17:35:07,316 | INFO | QA am: {'label': 'am', 'n': 5000, 'R_mae': 0.002672000226032806, 'R_rmse': 0.006505946291160472, 'T_mae': 0.0032916026268840796, 'T_rmse': 0.008654867242679434, 'phiR_mae': 0.02806459778707628, 'phiR_rmse': 0.12680624690114128, 'phiT_mae': 0.012741929187601081, 'phiT_rmse': 0.10299462844614637, 'R_out_of_01': 12, 'T_out_of_01': 8}
2026-03-29 17:35:07,318 | INFO | QA cr: {'label': 'cr', 'n': 5000, 'R_mae': 0.0028468198160631637, 'R_rmse': 0.006333117164604945, 'T_mae': 0.003076757986317825, 'T_rmse': 0.007645034749716436, 'phiR_mae': 0.02550039341327996, 'phiR_rmse': 0.10899355163259315, 'phiT_mae': 0.0158984526298179, 'phiT_rmse': 0.12893538264730103, 'R_ou

({'label': 'am',
  'n': 5000,
  'R_mae': 0.002672000226032806,
  'R_rmse': 0.006505946291160472,
  'T_mae': 0.0032916026268840796,
  'T_rmse': 0.008654867242679434,
  'phiR_mae': 0.02806459778707628,
  'phiR_rmse': 0.12680624690114128,
  'phiT_mae': 0.012741929187601081,
  'phiT_rmse': 0.10299462844614637,
  'R_out_of_01': 12,
  'T_out_of_01': 8},
 {'label': 'cr',
  'n': 5000,
  'R_mae': 0.0028468198160631637,
  'R_rmse': 0.006333117164604945,
  'T_mae': 0.003076757986317825,
  'T_rmse': 0.007645034749716436,
  'phiR_mae': 0.02550039341327996,
  'phiR_rmse': 0.10899355163259315,
  'phiT_mae': 0.0158984526298179,
  'phiT_rmse': 0.12893538264730103,
  'R_out_of_01': 15,
  'T_out_of_01': 3})

In [4]:
pos_test = np.array([9.81906933e+02, 8.67975732e+02, 9.41977049e+02, 9.27678994e+02,
 9.75982905e+02, 8.31890744e+02, 6.53923632e+02, 8.37959995e+02,
 9.21985370e+02, 7.74682001e+02, 9.93454774e+02, 7.25768577e+02,
 2.19316883e+02, 4.87712885e+02, 5.19326481e+02, 9.23125479e+02,
 4.85692989e+02, 5.12566172e+02, 7.41552996e+02, 7.16510746e+02,
 6.90332720e+02, 6.43673599e+02, 2.82473350e+02, 6.53451428e+01,
 3.17534948e+02, 1.56975911e+02, 5.98049070e+02, 3.64798669e+01,
 2.09424948e+02, 3.37403335e+02, 3.60524344e+02, 2.26015529e+02,
 3.01153404e+02, 4.72943603e+00, 6.25498466e+00, 8.00000000e-01,
 8.00000000e-01]
)

In [5]:
from pcm_pix.solution import save_preset

sol = save_preset(
    name="pos_test",
    pos=pos_test,
    cfg=CFG,
    sur0=sur0,
    sur1=sur1,
    preset_dir=CFG["preset_dir"],
    meta={"source": "test"},
)

In [6]:
"""
#Загрузить пресет для дальнейших расчётов/визуализации:
from pcm_pix.solution import load_preset

pos, cost, refl, trans, phi_refl, phi_trans, meta = load_preset("pos_test")

from pcm_pix.solution import Solution
from pathlib import Path
sol = Solution.from_json(Path(CFG["preset_dir"]) / "pos_test.json")
pos_m = sol.pos                       
R_am = sol.reflection["am"]
phi_R_am = sol.phase_shift_reflection["am"]
"""

'\n#Загрузить пресет для дальнейших расчётов/визуализации:\nfrom pcm_pix.solution import load_preset\n\npos, cost, refl, trans, phi_refl, phi_trans, meta = load_preset("pos_test")\n\nfrom pcm_pix.solution import Solution\nfrom pathlib import Path\nsol = Solution.from_json(Path(CFG["preset_dir"]) / "pos_test.json")\npos_m = sol.pos                       \nR_am = sol.reflection["am"]\nphi_R_am = sol.phase_shift_reflection["am"]\n'

In [7]:
#Проверить каталог пресетов, найти дубли и дополнить недостающие поля:
from pcm_pix.solution import check_and_fix_presets

report = check_and_fix_presets(
    cfg=CFG,
    sur0=sur0,
    sur1=sur1,
)

report

{'checked': 7,
 'fixed': [PosixPath('data/preset/pos_server_2026_03_27.json')],
 'skipped_no_surrogates': [],
 'duplicates': []}

In [8]:


# --- Presets / solutions catalog ---
from pcm_pix.solution import save_solution, load_solution

PRESETS = {
    "pos_2026_03_03_example": np.array([
        999.7153881712717, 989.8568611314538, 846.9306855895525, 812.3405141712682,
        357.50476722603656, 999.0189388651343, 853.5246592828234, 979.0165540829283,
        952.2077494627958, 998.2131906719599, 998.0020446893369, 638.3004244094977,
        645.2087664126163, 681.6157122198844, 702.1334382560799, 284.8875621006104,
        882.7549390819086, 520.3952331766168, 442.78456983718775, 875.627636396043,
        683.0421367034168, 725.5677666087739, 305.7428105275045, 302.3913087111928,
        246.68691437766532, 261.130662543832, 25.83901557205877, 397.619402446743,
        17.140592766392103, 140.12532478580698, 735.0979445623718, 211.69540821165236,
        355.55785007186574, 4.522892213428676, 2.827729567363648,
        0.9073471746931481, 0.8273584342733484
    ]),
    "pos_server_2026_03_04_night":np.array([
        9.99613315e+02, 8.86918846e+02, 6.06035395e+02, 8.54046947e+02,
        5.17645193e+02, 8.79810042e+02, 5.65420119e+02, 9.64383427e+02,
        9.97814618e+02, 9.55635573e+02, 9.95016420e+02, 5.99598526e+02,
        8.29227746e+02, 5.39970803e+02, 8.04025215e+02, 4.17264700e+02,
        6.15239459e+02, 3.62347946e+02, 4.20726459e+02, 6.09650053e+02,
        2.69023666e+02, 4.13247590e+02, 3.76852495e+02, 5.40484452e+02,
        1.99690690e+02, 3.57938384e+02, 4.35299951e+01, 2.57512612e+02,
        2.45768992e+02, 2.64399901e+02, 5.09649663e+02, 1.68530071e+02,
        2.33417540e+01, 4.73217432e+00, 3.35695242e+00, 9.56090488e-01,
        8.91457136e-01
    ]),
    "pos_test_2026_03_12": np.array([9.96841358e+02, 9.37302864e+02, 9.57294114e+02, 5.29040371e+02,
        9.77389395e+02, 8.58376681e+02, 8.42285256e+02, 9.33850847e+02,
        7.63770840e+02, 8.68523525e+02, 9.93136720e+02, 7.13942337e+02,
        1.72741142e+02, 4.06342614e+02, 3.83147986e+02, 6.38101254e+02,
        6.60311825e+02, 5.07564561e+02, 8.34434864e+02, 6.96096300e+02,
        6.80798657e+02, 6.45862244e+02, 2.87477815e+02, 7.04086071e+01,
        4.42449666e+01, 2.50460328e+02, 3.48349163e+02, 2.58011572e+02,
        7.10594600e+01, 3.19886408e+02, 2.33876373e+02, 2.56188765e+02,
        3.06062295e+02, 4.67935757e+00, 6.16194898e+00, 8.00000000e-01,
        8.00000000e-01]),
    "pos_server_2026_03_12": np.array([996.0443470708294, 572.8271927555644, 984.2487359207097, 887.5327249259412,
        881.9807993287004, 566.0181896913394, 876.0626657424411, 746.7439603540174, 
        804.525380579887, 901.4929955109988, 998.1095763354482, 724.4984270316959,
        106.6907199378395, 773.1027097258207, 477.17775717341044, 556.8891159329191,
        433.50739999462724, 577.3330032905027, 592.0556864456012, 711.9341242406858,
        685.0899094035707, 645.4252049086351, 318.94850557947257, 48.14530822881471, 
        574.9734674452097, 38.36816798416811, 161.8944738672979, 182.48062067122947, 
        199.78375691051463, 175.81728788584587, 272.8027577304986, 280.58320486862567, 
        312.024253921496, 4.6648036497638525, 0.0, 0.8, 
        0.8]),
    "pos_server_2026_03_27": np.array([9.99483818528431e-07, 9.188722265129985e-07, 8.242836509004236e-07, 7.28661893872381e-07, 8.148839867487604e-07, 9.561085381367339e-07, 8.582514907796472e-07, 7.737700037958455e-07, 8.106817330425298e-07, 7.321705799226643e-07, 9.984860188776006e-07, 6.449450445785125e-07, 6.826988850958929e-07, 7.124620752787625e-07, 5.627513193167834e-07, 4.99848140552402e-07, 6.043595974506708e-07, 5.038428965410576e-07, 5.604661990034843e-07, 5.638485399173609e-07, 1.3280927921633043e-07, 7.339901412359708e-07, 3.133460042266406e-07, 2.848064028829509e-07, 2.818830934747372e-07, 1.5869181919172922e-07, 2.2532964801650016e-08, 3.0428640206508813e-07, 7.864654602735334e-08, 1.749145959124581e-08, 5.7194056520267535e-08, 2.0853752152167705e-08, 3.4939702803051257e-07, 4.650226380466537, 3.1858791750474564, 0.8022855448310745, 0.8027976660422049]),
    "pos_server_2026_03_29": np.array([9.996308866151453e-07, 9.183935299627176e-07, 8.446424479162881e-07, 8.988312568140486e-07, 9.411160760495813e-07, 8.195237069878231e-07, 8.24034874151253e-07, 7.915650779962001e-07, 8.065231750425021e-07, 8.07848937297993e-07, 9.974010591892027e-07, 6.445686458313837e-07, 6.822223231909979e-07, 7.082805969464382e-07, 7.440978655402576e-07, 6.505807753847099e-07, 5.056165004252804e-07, 5.110472163809242e-07, 5.548702030688176e-07, 5.516781267211072e-07, 1.6162811206992207e-07, 7.355847854233951e-07, 3.12574886685502e-07, 2.841433612537425e-07, 2.8558267057684806e-07, 4.1182466495556076e-07, 3.351667217414924e-07, 5.159437442252511e-08, 5.14184099495392e-09, 2.0595476210369592e-08, 1.4229604933056378e-08, 2.0537993919273128e-08, 3.4632043070448966e-07, 4.678873902341533, 3.182262382798726, 0.8045215887790131, 0.8038624046598802]),

}

# save presets once (idempotent)
for name, pos in PRESETS.items():
    save_solution(run, name=name, pos=pos, cost=None, meta={"source": "preset"})

logger.info("presets saved to %s", run.results / "solutions")




2026-03-29 17:35:07,574 | INFO | saved solution pos_2026_03_03_example.json
2026-03-29 17:35:07,575 | INFO | saved solution pos_server_2026_03_04_night.json
2026-03-29 17:35:07,582 | INFO | saved solution pos_test_2026_03_12.json
2026-03-29 17:35:07,584 | INFO | saved solution pos_server_2026_03_12.json
2026-03-29 17:35:07,586 | INFO | saved solution pos_server_2026_03_27.json
2026-03-29 17:35:07,587 | INFO | saved solution pos_server_2026_03_29.json
2026-03-29 17:35:07,588 | INFO | presets saved to outputs/2026_03_12/results/solutions


In [10]:
from pcm_pix.optimize import load_hyperopt_params
loaded = load_hyperopt_params(CFG, run=run)
logger.info("hyperopt params loaded: %s", loaded)

2026-03-27 19:47:50,375 | INFO | Loaded hyperparams from data/hyperparams_final.json (17 keys)
2026-03-27 19:47:50,376 | INFO | hyperopt params loaded: True


In [14]:
from pcm_pix.hyperopt2 import resolve_hyperparams

resolve_hyperparams("pso", sur0, sur1, CFG, run, logger)

pos0 = PRESETS[CFG["preset_name"]]
resolve_hyperparams("de", sur0, sur1, CFG, run, logger, pos0=pos0)


2026-03-25 16:35:50,608 | INFO | Skip pso hyperopt (mode=use_saved)
2026-03-25 16:35:50,609 | INFO | Skip de hyperopt (mode=use_saved)


In [15]:
import numpy as np
import torch

from pcm_pix.optimize import (
    f_vec,
    run_pso,
    run_de_full,
    run_hybrid_pso_de,
)


def run_one(seed: int):
    # фиксируем сида для всего, что можем
    np.random.seed(seed)
    torch.manual_seed(seed)
    try:
        torch.cuda.manual_seed_all(seed)
    except Exception:
        pass

    logger.info("=== RUN seed=%s ===", seed)

    # тут ровно твой блок оптимизации, только возвращаем (pos, cost)
    if CFG["opt_mode"] == "preset":
        pos = PRESETS[CFG["preset_name"]]
        cost = float(f_vec(pos.reshape(1, -1), sur0, sur1, CFG)[0])
        logger.info("using preset %s cost=%.6f", CFG["preset_name"], cost)

    elif CFG["opt_mode"] == "pso":
        best = run_pso(sur0, sur1, CFG, run)
        pos, cost = best.pos, best.cost

    elif CFG["opt_mode"] == "de_full":
        pos0 = PRESETS[CFG["preset_name"]]
        best = run_de_full(sur0, sur1, CFG, run, pos=pos0)
        pos, cost = best.pos, best.cost

    else:
        best = run_hybrid_pso_de(sur0, sur1, CFG, run)
        pos, cost = best.pos, best.cost

    logger.info("FINAL cost=%.6f", float(cost))
    logger.info("FINAL pos_len=%s", len(pos))

    return np.array(pos), float(cost)

In [17]:
K = 10  # сколько прогонов
base_seed = SEED # базовый сид, можно взять из CFG или руками

best_cost = np.inf
best_pos = None
best_seed = None

for k in range(K):
    seed = base_seed + k
    pos_k, cost_k = run_one(seed)

    logger.info("RUN %s/%s seed=%s cost=%.6f", k + 1, K, seed, cost_k)

    if cost_k < best_cost:
        best_cost = cost_k
        best_pos = pos_k
        best_seed = seed

logger.info("=== BEST over %s runs: seed=%s cost=%.6f ===", K, best_seed, best_cost)
best_cost, best_seed

2026-03-25 16:42:31,339 | INFO | === RUN seed=69954 ===
2026-03-25 16:42:31,357 | INFO | PSO start: particles=3447 iters=347 dims=37
2026-03-25 16:42:31,361 - pyswarms.single.global_best - INFO - Optimize for 347 iters with {'c1': 1.4006010017715806, 'c2': 0.6479191118309222, 'w': 0.6003081724652297}
pyswarms.single.global_best:   0%|          |0/347

pyswarms.single.global_best: 100%|██████████|347/347, best_cost=2.5   
2026-03-25 16:43:05,869 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 2.4998856088656947, best pos: [5.28779691e-07 7.53692417e-07 5.72215066e-07 6.98376122e-07
 9.12873632e-07 5.81660417e-07 7.74714173e-07 5.81061070e-07
 9.77437402e-07 9.38863282e-07 6.02168054e-07 4.28348300e-07
 6.27904239e-07 4.63898553e-07 5.37780469e-07 5.87040536e-07
 4.31600434e-07 6.38618898e-07 4.26531587e-07 7.56728873e-07
 8.07592122e-07 3.79093482e-07 3.91607805e-08 1.06058377e-08
 6.21158720e-09 6.45364231e-08 2.69453933e-07 1.12504245e-07
 4.18446637e-07 2.82975655e-07 5.13905085e-07 5.84207548e-07
 1.56256515e-07 8.81443503e-01 3.54182874e+00 8.00911563e-01
 8.00003036e-01]
2026-03-25 16:43:05,870 | INFO | PSO done: cost=2.4998856088656947
2026-03-25 16:43:05,917 | INFO | DE start (full): maxiter=10000 popsize=22 init_mode=init_ar lc=True
2026-03-25 16:43:06,084 | INFO | Conv 0 2.828918
2026-03-25 16:43:0

KeyboardInterrupt: 

In [ ]:
# --- Save final solution to catalog ---
from pcm_pix.optimize import save_solution

name = f"final_{CFG['opt_mode']}"
path = save_solution(run, name=name, pos=best_pos, cost=float(best_cost), meta={"cfg": {"opt_mode": CFG["opt_mode"]}})
logger.info("final solution saved: %s", path.name)
path